# Multilingual UD & Phonological Dataset for Dependency Distance Analysis

This notebook demonstrates the **Multilingual UD & Phonological Density Dataset** — a curated collection of Universal Dependencies treebanks paired with phonological and typological features for cross-linguistic dependency-distance minimization research.

## What this artifact does

1. **Collects** UD treebank data from `commul/universal_dependencies` on HuggingFace for 39+ languages spanning 10 major families.
2. **Enriches** each language with phonological features (phoneme inventory sizes from PHOIBLE 2.0) and typological features (word order, family classification from WALS).
3. **Computes** dependency-distance statistics for every sentence: average distance, maximum distance, and distribution shapes.
4. **Enables** cross-linguistic comparison: Do SOV languages minimize dependency distances more than SVO? Do spoken registers show stronger minimization than written? Which families deviate from the universal pattern?

## Demo scope

This demo uses a curated subset of **5 languages from 4 families** (English, German, Finnish, Chinese, Japanese) with **3 sentences each** to illustrate the full analysis pipeline. The production run processes 39 languages with up to 200 sentences per treebank.

## Research context

Dependency Length Minimization (DLM) posits that human languages tend to keep syntactically related words close together, reducing processing cost. This dataset tests whether DLM interacts with:
- **Word order** (SVO vs SOV vs VSO)
- **Phonological density** (phoneme inventory size)
- **Register** (spoken vs written)
- **Language family** (are there family-specific deviations?)

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# No non-Colab packages needed for this demo — all dependencies are pre-installed on Colab.
# The dataset library is only needed for the full production run (loading from HuggingFace).
# This demo loads pre-extracted JSON data.

# Core packages — pre-installed on Colab, install locally to match Colab environment
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0', 'scipy==1.16.3', 'seaborn==0.13.2', 'scikit-learn==1.6.1')

In [ ]:
import json
import os
import urllib.request
from collections import defaultdict
from typing import Any, Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for headless execution
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

# NumPy 2.0 compatibility shims (for older packages that may still reference deprecated APIs)
if not hasattr(np, "alltrue"): np.alltrue = np.all
if not hasattr(np, "sometrue"): np.sometrue = np.any
if not hasattr(np, "product"): np.product = np.prod

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-45370e-phonotactic-constraint-on-dependency/main/round-1/dataset-1/demo/mini_demo_data.json"

def load_data() -> Dict[str, Any]:
    """Load demo data from GitHub URL with local fallback."""
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or local filesystem")


data = load_data()
print(f"Loaded dataset with {len(data['datasets'])} languages, {data['meta']['total_examples']} total examples")
print(f"Families: {data['meta']['families']}")
print(f"Word orders: {data['meta']['word_orders']}")

In [ ]:
# ── Configuration: ALL tunable parameters ──────────────────────────────────
# Start with ABSOLUTE MINIMUM values for the demo. Scale up in TODO 5.

# How many examples to use per dataset (language). Min=1, original=200.
N_EXAMPLES_PER_DATASET = 3

# How many datasets (languages) to include in analysis. Min=2, original=39.
N_DATASETS = 5

# Whether to compute per-family aggregations
COMPUTE_FAMILY_STATS = True

# Whether to compute per-word-order comparisons
COMPUTE_WORD_ORDER_STATS = True

# Whether to compute per-register (spoken vs written) comparisons
COMPUTE_REGISTER_STATS = True

# Whether to compute phonological density correlations
COMPUTE_PHONOLOGICAL_CORRELATION = True

# Number of bins for dependency distance histograms
HISTOGRAM_BINS = 10

# Figure size (width, height) in inches
FIG_SIZE = (14, 10)

# Seed for reproducibility
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)

print(f"Config: {N_DATASETS} datasets, {N_EXAMPLES_PER_DATASET} examples each")
print(f"Stats: family={COMPUTE_FAMILY_STATS}, word_order={COMPUTE_WORD_ORDER_STATS}, register={COMPUTE_REGISTER_STATS}")

## Data Parsing

Parse the JSON-encoded input/output fields from each example into structured data. The original `data.py` script stores tokens, heads, and dependency distances as JSON strings inside the `input` and `output` fields — we decode them here for analysis.

In [ ]:
def parse_examples(data: Dict[str, Any], n_per_dataset: int, n_datasets: int) -> List[Dict[str, Any]]:
    """Parse examples from the loaded data, limiting to n_per_dataset per language."""
    all_examples = []
    datasets = data['datasets'][:n_datasets]
    
    for ds in datasets:
        examples = ds['examples'][:n_per_dataset]
        for ex in examples:
            # Parse the JSON-encoded input
            input_data = json.loads(ex['input'])
            output_data = json.loads(ex['output'])
            
            parsed = {
                'language': ex.get('metadata_language', ''),
                'config_id': ex.get('metadata_config_id', ''),
                'family': ex.get('metadata_family', ''),
                'word_order': ex.get('metadata_word_order', ''),
                'register': ex.get('metadata_register', ''),
                'phoneme_count': ex.get('metadata_phoneme_count', 0),
                'syllable_complexity': ex.get('metadata_syllable_complexity', 0),
                'phonological_density': ex.get('metadata_phonological_density', 0),
                'tokens': input_data.get('tokens', []),
                'heads': input_data.get('head', []),
                'dependency_distances': input_data.get('dependency_distances', []),
                'avg_dep_dist': output_data.get('avg_dependency_distance', 0),
                'max_dep_dist': output_data.get('max_dependency_distance', 0),
                'n_deps': output_data.get('n_dependencies', 0),
                'sentence_text': output_data.get('sentence_text', ''),
            }
            all_examples.append(parsed)
    
    return all_examples


examples = parse_examples(data, N_EXAMPLES_PER_DATASET, N_DATASETS)
print(f"Parsed {len(examples)} examples from {N_DATASETS} languages")
print(f"\nFirst example:")
ex0 = examples[0]
print(f"  Language: {ex0['language']} ({ex0['word_order']})")
print(f"  Tokens: {ex0['tokens'][:10]}")
print(f"  Avg dep dist: {ex0['avg_dep_dist']:.3f}")
print(f"  Max dep dist: {ex0['max_dep_dist']}")
print(f"  N deps: {ex0['n_deps']}")

## Dependency Distance Statistics

Compute per-language and per-group statistics: mean, median, standard deviation of dependency distances. This is the core analysis — measuring how far apart syntactically related words tend to be in each language.

In [ ]:
def compute_language_stats(examples: List[Dict]) -> pd.DataFrame:
    """Compute dependency distance statistics per language."""
    lang_groups = defaultdict(list)
    for ex in examples:
        lang_groups[ex['language']].append(ex)
    
    rows = []
    for lang, exs in lang_groups.items():
        all_dists = []
        for ex in exs:
            all_dists.extend(ex['dependency_distances'])
        
        if not all_dists:
            continue
        
        rows.append({
            'language': lang,
            'family': exs[0]['family'],
            'word_order': exs[0]['word_order'],
            'register': exs[0]['register'],
            'phoneme_count': exs[0]['phoneme_count'],
            'phonological_density': exs[0]['phonological_density'],
            'n_sentences': len(exs),
            'n_dependencies': len(all_dists),
            'mean_dep_dist': np.mean(all_dists),
            'median_dep_dist': np.median(all_dists),
            'std_dep_dist': np.std(all_dists) if len(all_dists) > 1 else 0,
            'max_dep_dist': max(all_dists),
            'p25_dep_dist': np.percentile(all_dists, 25),
            'p75_dep_dist': np.percentile(all_dists, 75),
        })
    
    return pd.DataFrame(rows)


lang_stats = compute_language_stats(examples)
display(lang_stats[['language', 'family', 'word_order', 'n_sentences', 'mean_dep_dist', 'median_dep_dist', 'std_dep_dist', 'max_dep_dist']])

## Cross-Linguistic Comparison: Word Order Effects

Test the hypothesis that SOV languages exhibit different dependency distance patterns than SVO languages. This is a core prediction of Dependency Length Minimization theory — word order constraints may force longer dependencies in some languages.

In [ ]:
def compare_word_orders(examples: List[Dict]) -> Dict[str, Any]:
    """Compare dependency distance statistics across word orders."""
    wo_groups = defaultdict(list)
    for ex in examples:
        all_dists = ex['dependency_distances']
        if all_dists:
            wo_groups[ex['word_order']].extend(all_dists)
    
    results = {}
    for wo, dists in wo_groups.items():
        results[wo] = {
            'n': len(dists),
            'mean': float(np.mean(dists)),
            'median': float(np.median(dists)),
            'std': float(np.std(dists)),
            'min': float(min(dists)),
            'max': float(max(dists)),
        }
    
    # Statistical test if we have at least 2 groups
    wo_keys = list(results.keys())
    if len(wo_keys) >= 2:
        group1 = wo_groups[wo_keys[0]]
        group2 = wo_groups[wo_keys[1]]
        if len(group1) > 1 and len(group2) > 1:
            t_stat, p_value = stats.ttest_ind(group1, group2, equal_var=False)
            results['test'] = {
                'comparison': f"{wo_keys[0]} vs {wo_keys[1]}",
                't_statistic': float(t_stat),
                'p_value': float(p_value),
                'significant': p_value < 0.05,
            }
    
    return results


if COMPUTE_WORD_ORDER_STATS:
    wo_results = compare_word_orders(examples)
    print("=== Word Order Comparison ===")
    for wo, stats_data in wo_results.items():
        if wo == 'test':
            print(f"\nStatistical test: {stats_data['comparison']}")
            print(f"  t = {stats_data['t_statistic']:.4f}, p = {stats_data['p_value']:.4f}")
            print(f"  Significant (p < 0.05): {stats_data['significant']}")
        else:
            print(f"\n{wo}: n={stats_data['n']}, mean={stats_data['mean']:.3f}, "
                  f"median={stats_data['median']:.3f}, std={stats_data['std']:.3f}")

## Family-Level Analysis

Aggregate statistics by language family to identify family-specific patterns. Some families may show systematic deviations from the DLM prediction due to morphological richness or other structural properties.

In [ ]:
def compare_families(examples: List[Dict]) -> pd.DataFrame:
    """Compare dependency distance statistics across language families."""
    fam_groups = defaultdict(list)
    for ex in examples:
        fam_groups[ex['family']].extend(ex['dependency_distances'])
    
    rows = []
    for fam, dists in fam_groups.items():
        rows.append({
            'family': fam,
            'n_dependencies': len(dists),
            'mean_dep_dist': round(float(np.mean(dists)), 3),
            'median_dep_dist': round(float(np.median(dists)), 3),
            'std_dep_dist': round(float(np.std(dists)), 3),
            'max_dep_dist': int(max(dists)),
        })
    
    return pd.DataFrame(rows).sort_values('mean_dep_dist')


if COMPUTE_FAMILY_STATS:
    fam_stats = compare_families(examples)
    print("=== Family-Level Statistics ===")
    display(fam_stats)

## Phonological Density Correlation

Test whether phonological density (phoneme inventory size normalized by syllable complexity) correlates with dependency distance patterns. This is the novel hypothesis of this research: languages with richer phonological inventories may have different syntactic compression strategies.

In [ ]:
def compute_phonological_correlation(examples: List[Dict]) -> Dict[str, Any]:
    """Compute correlation between phonological density and dependency distances."""
    # Aggregate per language
    lang_data = defaultdict(lambda: {'dists': [], 'phon_density': 0, 'phoneme_count': 0})
    for ex in examples:
        lang_data[ex['language']]['dists'].extend(ex['dependency_distances'])
        lang_data[ex['language']]['phon_density'] = ex['phonological_density']
        lang_data[ex['language']]['phoneme_count'] = ex['phoneme_count']
    
    means = []
    densities = []
    phoneme_counts = []
    
    for lang, data in lang_data.items():
        if data['dists']:
            means.append(np.mean(data['dists']))
            densities.append(data['phon_density'])
            phoneme_counts.append(data['phoneme_count'])
    
    results = {}
    if len(means) >= 3:
        r_density, p_density = stats.pearsonr(densities, means)
        r_phoneme, p_phoneme = stats.pearsonr(phoneme_counts, means)
        results['phonological_density_correlation'] = {
            'r': float(r_density),
            'p': float(p_density),
            'n_languages': len(means),
        }
        results['phoneme_count_correlation'] = {
            'r': float(r_phoneme),
            'p': float(p_phoneme),
            'n_languages': len(means),
        }
    else:
        results['warning'] = f"Need at least 3 languages for correlation, have {len(means)}"
    
    return results


if COMPUTE_PHONOLOGICAL_CORRELATION:
    phono_results = compute_phonological_correlation(examples)
    print("=== Phonological Density Correlation ===")
    for key, val in phono_results.items():
        if key == 'warning':
            print(f"  ⚠ {val}")
        else:
            print(f"  {key}: r = {val['r']:.4f}, p = {val['p']:.4f} (n = {val['n_languages']})")

## Visualization

Generate comprehensive figures showing: (1) dependency distance distributions by language, (2) word order comparison, (3) family-level patterns, and (4) phonological density scatter plot.

In [ ]:
def plot_all_results(examples: List[Dict], lang_stats: pd.DataFrame) -> None:
    """Generate comprehensive visualization of dependency distance analysis."""
    fig = plt.figure(figsize=FIG_SIZE)
    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.35, wspace=0.3)
    
    # ── Panel A: Dependency distance distributions by language ──
    ax1 = fig.add_subplot(gs[0, 0])
    lang_groups = defaultdict(list)
    for ex in examples:
        lang_groups[ex['language']].extend(ex['dependency_distances'])
    
    colors = plt.cm.tab10(np.linspace(0, 1, len(lang_groups)))
    for i, (lang, dists) in enumerate(sorted(lang_groups.items())):
        ax1.hist(dists, bins=HISTOGRAM_BINS, alpha=0.6, label=lang, color=colors[i], density=True, edgecolor='black', linewidth=0.5)
    ax1.set_xlabel('Dependency Distance')
    ax1.set_ylabel('Density')
    ax1.set_title('A: Dependency Distance Distributions')
    ax1.legend(fontsize=8, loc='upper right')
    ax1.grid(alpha=0.3)
    
    # ── Panel B: Mean dependency distance by word order ──
    ax2 = fig.add_subplot(gs[0, 1])
    wo_means = {}
    wo_all = defaultdict(list)
    for ex in examples:
        wo_all[ex['word_order']].extend(ex['dependency_distances'])
    for wo, dists in wo_all.items():
        wo_means[wo] = np.mean(dists)
    
    wo_labels = list(wo_means.keys())
    wo_values = [wo_means[w] for w in wo_labels]
    wo_colors = ['#2196F3' if w == 'SVO' else '#FF5722' for w in wo_labels]
    bars = ax2.bar(wo_labels, wo_values, color=wo_colors, edgecolor='black', alpha=0.8)
    for bar, val in zip(bars, wo_values):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, 
                 f'{val:.2f}', ha='center', fontsize=10, fontweight='bold')
    ax2.set_ylabel('Mean Dependency Distance')
    ax2.set_title('B: Mean Distance by Word Order')
    ax2.grid(axis='y', alpha=0.3)
    
    # ── Panel C: Family comparison boxplot ──
    ax3 = fig.add_subplot(gs[1, 0])
    fam_data = defaultdict(list)
    for ex in examples:
        fam_data[ex['family']].extend(ex['dependency_distances'])
    
    fam_labels = sorted(fam_data.keys())
    fam_values = [fam_data[f] for f in fam_labels]
    bp = ax3.boxplot(fam_values, labels=fam_labels, patch_artist=True)
    fam_colors = plt.cm.Set2(np.linspace(0, 1, len(fam_labels)))
    for patch, color in zip(bp['boxes'], fam_colors):
        patch.set_facecolor(color)
        patch.set_edgecolor('black')
    ax3.set_ylabel('Dependency Distance')
    ax3.set_title('C: Distance Distribution by Family')
    ax3.tick_params(axis='x', rotation=15)
    ax3.grid(axis='y', alpha=0.3)
    
    # ── Panel D: Phonological density vs mean dependency distance ──
    ax4 = fig.add_subplot(gs[1, 1])
    lang_means = {}
    lang_densities = {}
    for ex in examples:
        lang = ex['language']
        if lang not in lang_means:
            lang_means[lang] = []
            lang_densities[lang] = ex['phonological_density']
        lang_means[lang].extend(ex['dependency_distances'])
    
    x_vals = [lang_densities[l] for l in lang_means]
    y_vals = [np.mean(lang_means[l]) for l in lang_means]
    labels = list(lang_means.keys())
    
    sc = ax4.scatter(x_vals, y_vals, s=100, c=colors[:len(labels)], edgecolors='black', zorder=3)
    for i, (x, y, label) in enumerate(zip(x_vals, y_vals, labels)):
        ax4.annotate(label, (x, y), textcoords="offset points", xytext=(5, 5), fontsize=8)
    
    if len(x_vals) >= 3:
        z = np.polyfit(x_vals, y_vals, 1)
        p = np.poly1d(z)
        x_line = np.linspace(min(x_vals), max(x_vals), 100)
        ax4.plot(x_line, p(x_line), 'r--', alpha=0.7, linewidth=2, label=f'Fit: y={z[0]:.3f}x+{z[1]:.3f}')
        ax4.legend(fontsize=8)
    
    ax4.set_xlabel('Phonological Density')
    ax4.set_ylabel('Mean Dependency Distance')
    ax4.set_title('D: Phonological Density vs Dependency Distance')
    ax4.grid(alpha=0.3)
    
    plt.suptitle('Dependency Distance Minimization: Cross-Linguistic Analysis', fontsize=14, fontweight='bold', y=1.02)
    plt.savefig('dependency_distance_analysis.png', dpi=150, bbox_inches='tight')
    plt.savefig('dependency_distance_analysis.pdf', format='pdf', bbox_inches='tight')
    plt.close()
    print("Saved: dependency_distance_analysis.png and .pdf")


plot_all_results(examples, lang_stats)

## Summary Table

Print a comprehensive summary of all computed statistics in a readable format.

In [ ]:
print("=" * 80)
print("DEPENDENCY DISTANCE MINIMIZATION: CROSS-LINGUISTIC ANALYSIS — SUMMARY")
print("=" * 80)

print(f"\nDataset: {data['meta']['title']}")
print(f"Languages analyzed: {len(lang_stats)}")
print(f"Total sentences: {lang_stats['n_sentences'].sum()}")
print(f"Total dependencies: {lang_stats['n_dependencies'].sum()}")

print("\n" + "-" * 80)
print("PER-LANGUAGE STATISTICS")
print("-" * 80)
print(f"{'Language':<15} {'Family':<18} {'WO':<5} {'Sent':>4} {'Mean':>6} {'Median':>7} {'Std':>6} {'Max':>4}")
print("-" * 80)
for _, row in lang_stats.iterrows():
    print(f"{row['language']:<15} {row['family']:<18} {row['word_order']:<5} "
          f"{row['n_sentences']:>4} {row['mean_dep_dist']:>6.3f} {row['median_dep_dist']:>7.3f} "
          f"{row['std_dep_dist']:>6.3f} {row['max_dep_dist']:>4}")

if COMPUTE_WORD_ORDER_STATS:
    print("\n" + "-" * 80)
    print("WORD ORDER COMPARISON")
    print("-" * 80)
    for wo, stats_data in wo_results.items():
        if wo == 'test':
            print(f"Test: {stats_data['comparison']}")
            print(f"  t = {stats_data['t_statistic']:.4f}, p = {stats_data['p_value']:.4f}")
            sig = "YES" if stats_data['significant'] else "NO"
            print(f"  Significant (p < 0.05): {sig}")
        else:
            print(f"{wo}: n={stats_data['n']}, mean={stats_data['mean']:.3f}, median={stats_data['median']:.3f}, std={stats_data['std']:.3f}")

if COMPUTE_FAMILY_STATS:
    print("\n" + "-" * 80)
    print("FAMILY-LEVEL STATISTICS")
    print("-" * 80)
    print(fam_stats.to_string(index=False))

if COMPUTE_PHONOLOGICAL_CORRELATION:
    print("\n" + "-" * 80)
    print("PHONOLOGICAL DENSITY CORRELATION")
    print("-" * 80)
    for key, val in phono_results.items():
        if key == 'warning':
            print(f"  ⚠ {val}")
        else:
            print(f"  {key}: r = {val['r']:.4f}, p = {val['p']:.4f} (n = {val['n_languages']})")

print("\n" + "=" * 80)
print("NOTE: This is a demo with minimal data (3 sentences per language).")
print("The full production run processes 39 languages with up to 200 sentences each.")
print("=" * 80)